🎯 **You Sank My Function** — Black-Box Optimisation Capstone · Imperial College Business School

**Weekly pipeline** · [04 Consolidate](../pipeline/04_consolidate_data.ipynb) → [05 Suggest Engine](../pipeline/05_suggest_engine.ipynb) → [06 Diagnose](../pipeline/06_diagnose.ipynb)
**Tutorials** · [01 GP theory](../tutorials/01_gp_bayesian_optimization_tutorial.ipynb) · [02 Acquisition engine](../tutorials/02_gp_suggest_input_tutorial.ipynb) · [03 EDA & prototype](../tutorials/03_capstone_eda_and_prototype.ipynb)
**Diagnostics** · [A1 SVM](../diagnostics/A1_svm_analysis.ipynb) · [A2 Logistic regression](../diagnostics/A2_logistic_regression.ipynb) · [A3 Landscape gallery](../diagnostics/A3_landscape_gallery.ipynb)

> 📍 **This notebook** — **Phase 3 of the weekly pipeline.** The gated, two-tier diagnostic router: cheap structural signals first, targeted confirmatory tests only when warranted. Audits whether each function's current method is still supported by the data.

---

# 06 · Diagnostic Router — *You Sank My Function*
### Guided, not brute-force

This notebook turns the **diagnose-first** methodology into code. It does **not**
fit every benchmark and model to every function (which, at 10–40 points in 2–8D,
would manufacture false confidence). Instead it works in two gated stages:

- **Tier 0 — cheap structural signals** (little/no fitting) that *scope* each
  function: needle-ness, noise regime, learnability, separability, per-dim
  sensitivity/direction.
- **Tier 1 — targeted confirmatory tests** run **only** when a Tier-0 signal or
  the function's prior hypothesis points at them: benchmark-identity
  *falsification* (against a *named* candidate, never a library sweep), a
  cross-validated separable fit, or source inversion.

Every verdict carries an evidence trail and is allowed to be
**"insufficient evidence — keep current method."** A `redirect` flag fires when
the data contradicts the prior (the F3 case). The router also **audits** the
current per-function record: does the evidence still support the method in use?


## 0 · Config, data, and priors (the guiding hypotheses)

In [1]:
import warnings, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

obs  = pd.read_csv("consolidated_observations.csv")
DIMS = obs.groupby("function").dim.first().to_dict()

# Gate thresholds (the skepticism dial — looser suggests more, stricter forces
# more 'insufficient' verdicts).
GATES = dict(needle_decades=6.0, needle_bg=0.50,
             bench_confirm_rho=0.95, bench_confirm_r2=0.99, bench_reject_rho=0.50,
             separable_r2=0.80, separable_gap=0.10,
             learn_high=0.70, learn_mod=0.30,
             noise_free=0.01, noise_low=0.10, sens_dir=0.30)

# PRIOR per function = the guiding hypothesis + a NAMED benchmark candidate to
# test (only [0,1]-native benchmarks get an exact recompute) + the current method
# (so we can audit it). This is the 'supervision' that keeps scope narrow.
PRIORS = {
 1: dict(hyp="radiation field / hidden source", benchmark=None,           current="source localisation (ZoMBI/inversion)"),
 2: dict(hyp="single high-x2 ridge, noisy",      benchmark=None,           current="manual ridge inference"),
 3: dict(hyp="Hartmann-3D? (inherited label)",   benchmark="hartmann3",    current="exploit benchmark point (identity unconfirmed)"),
 4: dict(hyp="smooth single basin",              benchmark=None,           current="GP-BO (EI)"),
 5: dict(hyp="unimodal, Rosenbrock fingerprint", benchmark=None,           current="log-GP posterior-mean"),
 6: dict(hyp="separable recipe (ST rejected)",   benchmark=None,           current="separable polynomial inversion"),
 7: dict(hyp="Hartmann-6D",                      benchmark="hartmann6",    current="analytic solve (Hartmann-6D)"),
 8: dict(hyp="separable weighted quadratic",     benchmark=None,           current="analytic solve (quadratic)"),
}
print("Loaded", len(obs), "observations · priors for", len(PRIORS), "functions")

Loaded 279 observations · priors for 8 functions


## 1 · Benchmark library (for *targeted* identity tests only)

Only the benchmarks defined natively on [0,1]ᵈ get an exact recompute, because
those are the only ones we can test without also guessing an input rescaling.
A named candidate is recomputed on the real inputs; the test is a rank
correlation plus an output-affine residual fit — robust to unknown scaling.

In [2]:
_H3_A = np.array([[3.,10,30],[.1,10,35],[3.,10,30],[.1,10,35]])
_H3_P = 1e-4*np.array([[3689,1170,2673],[4699,4387,7470],[1091,8732,5547],[381,5743,8828]])
_H6_A = np.array([[10,3,17,3.5,1.7,8],[.05,10,17,.1,8,14],[3,3.5,1.7,10,17,8],[17,8,.05,10,.1,14]])
_H6_P = 1e-4*np.array([[1312,1696,5569,124,8283,5886],[2329,4135,8307,3736,1004,9991],
                       [2348,1451,3522,2883,3047,6650],[4047,8828,8732,5743,1091,381]])
_ALPHA = np.array([1.0,1.2,3.0,3.2])

def _hartmann(X, A, P):
    X = np.atleast_2d(X)
    out = np.zeros(len(X))
    for i in range(4):
        out += _ALPHA[i]*np.exp(-np.sum(A[i]*(X - P[i])**2, axis=1))
    return out                      # positive (maximisation) form

BENCHMARKS = {
    "hartmann3": lambda X: _hartmann(X, _H3_A, _H3_P),
    "hartmann6": lambda X: _hartmann(X, _H6_A, _H6_P),
}
print("Benchmark library:", list(BENCHMARKS))

Benchmark library: ['hartmann3', 'hartmann6']


## 2 · Tier-0 — cheap structural signals (scope the problem)

In [3]:
def _gp_cv_r2(X, y, ls=0.2):
    n = len(y)
    if n < 6: return np.nan
    k = C(1.0)*Matern(ls, nu=2.5) + WhiteKernel(1e-3)
    gp = GaussianProcessRegressor(kernel=k, normalize_y=True, n_restarts_optimizer=1)
    cv = KFold(n_splits=min(5, n//2), shuffle=True, random_state=0)
    return float(np.mean(cross_val_score(gp, X, y, cv=cv, scoring="r2")))

def sig_needle(y):
    ay = np.abs(y); eps = 1e-300
    decades = float(np.log10(ay.max()+eps) - np.log10(np.median(ay)+eps))
    bg = float(np.mean(ay < 0.01*ay.max()))
    return dict(decades=round(decades,1), background=round(bg,2),
                is_needle=(decades >= GATES["needle_decades"] and bg >= GATES["needle_bg"]))

def sig_noise(X, y):
    n = len(y)
    if n < 6: return dict(noise_ratio=np.nan, regime="unknown")
    k = C(1.0)*Matern(0.2, nu=2.5) + WhiteKernel(noise_level=1e-3,
                                                 noise_level_bounds=(1e-10,1e2))
    gp = GaussianProcessRegressor(kernel=k, normalize_y=True, n_restarts_optimizer=2).fit(X, y)
    noise_sd = np.sqrt(gp.kernel_.k2.noise_level)
    ratio = float(noise_sd/(np.std(y)+1e-12))
    regime = ("noise-free" if ratio < GATES["noise_free"] else
              "low-noise" if ratio < GATES["noise_low"] else "noisy")
    return dict(noise_ratio=round(ratio,3), regime=regime)

def sig_learnability(X, y):
    r2 = _gp_cv_r2(X, y)
    tier = ("unknown" if not np.isfinite(r2) else
            "high" if r2 >= GATES["learn_high"] else
            "moderate" if r2 >= GATES["learn_mod"] else "low")
    return dict(gp_cv_r2=round(r2,2) if np.isfinite(r2) else None, tier=tier)

def _additive_features(X):
    cols = [np.ones((len(X),1))]
    for j in range(X.shape[1]):
        x = X[:, [j]]; cols += [x, x**2, x**3]
    return np.hstack(cols)                         # no cross terms => additive

def sig_separability(X, y):
    n = len(y)
    if n < 8: return dict(add_cv_r2=None, full_cv_r2=None, separable=False)
    cv = KFold(n_splits=min(5, n//2), shuffle=True, random_state=0)
    add_r2 = float(np.mean(cross_val_score(Ridge(alpha=1.0),
                    _additive_features(X), y, cv=cv, scoring="r2")))
    full_r2 = _gp_cv_r2(X, y)
    gap = (full_r2 - add_r2) if np.isfinite(full_r2) else 0.0
    sep = (add_r2 >= GATES["separable_r2"] and gap <= GATES["separable_gap"])
    return dict(add_cv_r2=round(add_r2,2), full_cv_r2=round(full_r2,2) if np.isfinite(full_r2) else None,
                separable=bool(sep))

def sig_sensitivity(X, y):
    dirs = []
    for j in range(X.shape[1]):
        r, _ = spearmanr(X[:, j], y); r = 0.0 if np.isnan(r) else r
        d = "↑" if r >= GATES["sens_dir"] else "↓" if r <= -GATES["sens_dir"] else "·"
        dirs.append(f"x{j+1}{d}")
    return dict(directions=" ".join(dirs))

## 3 · Tier-1 — targeted confirmatory tests (gated)

In [4]:
def test_benchmark(X, y, name):
    """Falsify/confirm a NAMED [0,1]-native benchmark via rank corr + affine residual."""
    raw = BENCHMARKS[name](X)
    rho, _ = spearmanr(raw, y); rho = 0.0 if np.isnan(rho) else float(rho)
    lin = LinearRegression().fit(raw.reshape(-1,1), y)
    r2 = float(r2_score(y, lin.predict(raw.reshape(-1,1))))
    if abs(rho) >= GATES["bench_confirm_rho"] and r2 >= GATES["bench_confirm_r2"]:
        verdict = "CONFIRMED"
    elif abs(rho) < GATES["bench_reject_rho"]:
        verdict = "REJECTED"
    else:
        verdict = "inconclusive"
    return dict(name=name, rho=round(rho,3), affine_r2=round(r2,3), verdict=verdict)

def test_separable_fit(X, y):
    """Per-dim Ridge polynomial; recover per-dim optima; report CV R²."""
    cv = KFold(n_splits=min(5, len(y)//2), shuffle=True, random_state=0)
    cvr2 = float(np.mean(cross_val_score(Ridge(alpha=1.0), _additive_features(X),
                                         y, cv=cv, scoring="r2")))
    model = Ridge(alpha=1.0).fit(_additive_features(X), y)
    grid = np.linspace(0, 1, 201)
    optima = []
    for j in range(X.shape[1]):
        Xg = np.tile(np.mean(X, axis=0), (201, 1)); Xg[:, j] = grid
        optima.append(round(float(grid[np.argmax(model.predict(_additive_features(Xg)))]), 3))
    return dict(cv_r2=round(cvr2,2), per_dim_optimum=optima,
                accept=bool(cvr2 >= GATES["separable_r2"]))

def test_source_inversion(X, y):
    """Log-space Gaussian decay (Ridge): recover source + fit R²."""
    ay = np.abs(y); mask = ay > ay.max()*1e-12
    if mask.sum() < X.shape[1] + 2:
        return dict(source=None, r2=None, note="too few informative points")
    logy = np.log(ay[mask] + 1e-300); Xm = X[mask]
    Phi = np.column_stack([Xm, (Xm**2).sum(axis=1)])
    reg = Ridge(alpha=0.01).fit(Phi, logy)
    r2 = float(reg.score(Phi, logy)); b = reg.coef_; b3 = b[-1]
    if b3 >= 0:
        return dict(source=None, r2=round(r2,2), note="non-physical (b3>=0)")
    sigma2 = -1.0/(2*b3); src = (b[:-1]*sigma2)
    return dict(source=[round(float(v),3) for v in src], r2=round(r2,2),
                note="ok" if np.all((src>=-0.1)&(src<=1.1)) else "source near/outside box")

## 4 · The router

Order of resolution: needle → named-benchmark test → separability → fall back to
GP-BO, with an "insufficient evidence" exit. Each branch records why it fired and
sets a `redirect` flag if it contradicts the prior.

In [5]:
def diagnose(fid):
    d = DIMS[fid]; pri = PRIORS[fid]
    sub = obs[obs.function == fid]
    X = sub[[f"x{i+1}" for i in range(d)]].to_numpy(float); y = sub["y"].to_numpy(float)

    t0 = dict(needle=sig_needle(y), noise=sig_noise(X, y),
              learn=sig_learnability(X, y), sep=sig_separability(X, y),
              sens=sig_sensitivity(X, y))
    ev, redirect, t1 = [], False, {}

    # 1) needle -> source inversion
    if t0["needle"]["is_needle"]:
        si = test_source_inversion(X, y); t1["source"] = si
        rec = "source localisation (ZoMBI / inversion)"
        conf = "high" if (si["r2"] or 0) > 0.5 else "medium"
        ev.append(f"needle: {t0['needle']['decades']} decades, "
                  f"{int(t0['needle']['background']*100)}% background")
    # 2) named benchmark -> falsification
    elif pri["benchmark"]:
        bf = test_benchmark(X, y, pri["benchmark"]); t1["benchmark"] = bf
        ev.append(f"benchmark {bf['name']}: rho={bf['rho']}, affine_R2={bf['affine_r2']} -> {bf['verdict']}")
        if bf["verdict"] == "CONFIRMED":
            rec, conf = f"analytic solve ({bf['name']} confirmed)", "high"
        elif bf["verdict"] == "REJECTED":
            rec, conf, redirect = "benchmark refuted -> structural/BO fallback", "low", True
        else:
            rec, conf = "inconclusive -> keep current method", "low"
    # 3) separability -> separable fit
    elif t0["sep"]["separable"]:
        sf = test_separable_fit(X, y); t1["separable"] = sf
        ev.append(f"separable: additive CV R²={t0['sep']['add_cv_r2']} (gap small)")
        if sf["accept"]:
            rec, conf = "separable inversion (per-dim analytic optima)", \
                        "high" if sf["cv_r2"] >= 0.9 else "medium"
        else:
            rec, conf = "GP-BO (separable fit weak on CV)", "low"
    # 4) fall back to GP-BO, gated by learnability
    else:
        tier = t0["learn"]["tier"]
        if tier in ("high", "moderate"):
            rec = "GP-BO (EI; mean-exploitation if huge range; regime: " + t0["noise"]["regime"] + ")"
            conf = "medium" if tier == "high" else "low"
            ev.append(f"learnable (GP CV R²={t0['learn']['gp_cv_r2']}, {tier})")
        else:
            rec, conf = "INSUFFICIENT EVIDENCE — explore / keep current method", "low"
            ev.append(f"not modellable at this density (GP CV R²={t0['learn']['gp_cv_r2']})")

    return dict(fid=fid, recommended=rec, confidence=conf, redirect=redirect,
                regime=t0["noise"]["regime"], directions=t0["sens"]["directions"],
                evidence="; ".join(ev), current=pri["current"], tier0=t0, tier1=t1)

## 5 · Run the router on all eight + audit against current records

In [6]:
results = [diagnose(f) for f in range(1, 9)]
report = pd.DataFrame([{
    "F": f"F{r['fid']}", "recommended": r["recommended"], "confidence": r["confidence"],
    "redirect?": "YES" if r["redirect"] else "", "regime": r["regime"],
    "directions": r["directions"], "current_method": r["current"],
} for r in results])
pd.set_option("display.max_colwidth", 70)
report

,F,recommended,confidence,redirect?,regime,directions,current_method
0,F1,INSUFFICIENT EVIDENCE — explore / keep current method,low,,noise-free,x1↑ x2·,source localisation (ZoMBI/inversion)
1,F2,GP-BO (EI; mean-exploitation if huge range; regime: noisy),low,,noisy,x1↑ x2·,manual ridge inference
2,F3,benchmark refuted -> structural/BO fallback,low,YES,noisy,x1· x2· x3·,exploit benchmark point (identity unconfirmed)
3,F4,GP-BO (EI; mean-exploitation if huge range; regime: noise-free),medium,,noise-free,x1↓ x2↓ x3· x4↓,GP-BO (EI)
4,F5,GP-BO (EI; mean-exploitation if huge range; regime: noise-free),medium,,noise-free,x1· x2↑ x3↑ x4↑,log-GP posterior-mean
5,F6,GP-BO (EI; mean-exploitation if huge range; regime: noisy),medium,,noisy,x1· x2· x3↑ x4↑ x5↓,separable polynomial inversion
6,F7,analytic solve (hartmann6 confirmed),high,,noise-free,x1↓ x2↓ x3· x4· x5↓ x6↑,analytic solve (Hartmann-6D)
7,F8,separable inversion (per-dim analytic optima),high,,noise-free,x1↓ x2↓ x3↓ x4↓ x5↑ x6· x7↓ x8·,analytic solve (quadratic)


In [7]:
print("EVIDENCE TRAIL\n" + "="*70)
for r in results:
    flag = "  [REDIRECT]" if r["redirect"] else ""
    print(f"\nF{r['fid']} -> {r['recommended']}  ({r['confidence']}){flag}")
    print(f"   evidence: {r['evidence']}")

EVIDENCE TRAIL

F1 -> INSUFFICIENT EVIDENCE — explore / keep current method  (low)
   evidence: not modellable at this density (GP CV R²=0.02)

F2 -> GP-BO (EI; mean-exploitation if huge range; regime: noisy)  (low)
   evidence: learnable (GP CV R²=0.35, moderate)

F3 -> benchmark refuted -> structural/BO fallback  (low)  [REDIRECT]
   evidence: benchmark hartmann3: rho=0.23, affine_R2=0.012 -> REJECTED

F4 -> GP-BO (EI; mean-exploitation if huge range; regime: noise-free)  (medium)
   evidence: learnable (GP CV R²=0.93, high)

F5 -> GP-BO (EI; mean-exploitation if huge range; regime: noise-free)  (medium)
   evidence: learnable (GP CV R²=0.96, high)

F6 -> GP-BO (EI; mean-exploitation if huge range; regime: noisy)  (medium)
   evidence: learnable (GP CV R²=0.85, high)

F7 -> analytic solve (hartmann6 confirmed)  (high)
   evidence: benchmark hartmann6: rho=1.0, affine_R2=1.0 -> CONFIRMED

F8 -> separable inversion (per-dim analytic optima)  (high)
   evidence: separable: additive CV R

In [8]:
# Audit: does the diagnostic agree with the method currently in use?
print("AUDIT — diagnostic vs current record\n" + "="*70)
for r in results:
    rec, cur = r["recommended"].lower(), r["current"].lower()
    agree = (("source" in rec and "source" in cur) or
             ("analytic" in rec and "analytic" in cur) or
             ("separable" in rec and "separable" in cur) or
             ("gp-bo" in rec and ("gp" in cur or "bo" in cur or "mean" in cur)))
    tag = "agrees" if agree else ("REFUTES PRIOR" if r["redirect"] else "differs — review")
    print(f"  F{r['fid']:<2} {tag:<14} | diag: {r['recommended'][:42]:<42} | current: {r['current']}")

AUDIT — diagnostic vs current record
  F1  differs — review | diag: INSUFFICIENT EVIDENCE — explore / keep cur | current: source localisation (ZoMBI/inversion)
  F2  differs — review | diag: GP-BO (EI; mean-exploitation if huge range | current: manual ridge inference
  F3  REFUTES PRIOR  | diag: benchmark refuted -> structural/BO fallbac | current: exploit benchmark point (identity unconfirmed)
  F4  agrees         | diag: GP-BO (EI; mean-exploitation if huge range | current: GP-BO (EI)
  F5  agrees         | diag: GP-BO (EI; mean-exploitation if huge range | current: log-GP posterior-mean
  F6  differs — review | diag: GP-BO (EI; mean-exploitation if huge range | current: separable polynomial inversion
  F7  agrees         | diag: analytic solve (hartmann6 confirmed)       | current: analytic solve (Hartmann-6D)
  F8  agrees         | diag: separable inversion (per-dim analytic opti | current: analytic solve (quadratic)


## 6 · How this stays guided (not brute force)

Three mechanisms keep the scope narrow:

1. **Priors do the steering.** Benchmark identity is only ever tested against a
   *named* candidate carried in `PRIORS` — never a library sweep — so there is no
   multiple-comparisons fishing.
2. **Tier-1 fits are gated by Tier-0 signals.** A separable model is only fitted
   when the cheap additive-vs-full signal already says "separable"; source
   inversion only when the needle signal fires. Cross-validation, not in-sample
   fit, decides acceptance.
3. **"Insufficient evidence" is a first-class verdict.** When 32 points in 4D
   cannot resolve structure, the router says so instead of inventing a method.

**Extending it:** add a benchmark to `BENCHMARKS` + name it in a function's
`PRIORS`; or add a Tier-0 signal and a gated Tier-1 test. The `GATES` dict is the
single skepticism dial. The router is the post-supervision layer: it never picks
the weekly query (that stays with the per-function methods) — it audits whether
the method in use is still supported by the data, and flags redirects.


## 7 · Export

In [9]:
report.to_csv("diagnosis_report.csv", index=False)
print("Wrote:", "diagnosis_report.csv")

Wrote: diagnosis_report.csv
